## Algoritmos seleccionados
### Regresión Lineal, Random Forest, Support Vector Machine

## Error
#### RMSE para Regresion Lineal
#### Accuracy para Random Forest
#### Recall para SVM

## Tipos de datos

### Variables independientes
**Variables predictoras que se usan como entrada en los modelos:**
- Knee_Strength_Score  
- Hamstring_Flexibility  
- Reaction_Time_ms  
- Balance_Test_Score  
- Sprint_Speed_10m_s  
- Agility_Score  
- Stress_Level_Score  
### Factores controlables
**Pueden modificarse mediante entrenamiento, hábitos o rutinas:**
- Sleep_Hours_Per_Night  
- Nutrition_Quality_Score  
- Warmup_Routine_Adherence  
### Factores no controlables
**Características fijas o difíciles de alterar, pero que influyen en el riesgo de lesión:**
- Height_cm  
- Previous_Injury_Count  
- Position_Goalkeeper  
- Position_Midfielder  
### Variable dependiente
**Resultado que se busca predecir:**
- `injury_next_season` → Indica si el jugador se lesionará la próxima temporada  
  (`1 = Sí`, `0 = No`)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("test_estandarizado_percentiles.csv")
target = "TestScore_Math"


y = df[target]
x = df.drop(columns = [target])


x_train_val, x_test, y_train_val, y_test = train_test_split(x,y, test_size = 0.2, random_state = 42)

x_train, x_val, y_train, y_val = train_test_split(x_train_val,y_train_val, test_size = 0.2, random_state = 42)


print(f"Train : {len(x_train_val)}")
print(f"Validation : {len(x_val)}")
print(f"Test : {len(x_test)}")

x_train.to_csv("X_train.csv", index=False)
y_train.to_csv("Y_train.csv", index=False)
x_val.to_csv("X_val.csv", index=False)
y_val.to_csv("Y_val.csv", index=False)
x_test.to_csv("X_test.csv", index=False)
y_test.to_csv("Y_test.csv", index=False)

trainX = pd.read_csv("X_train.csv")
trainY = np.ravel(pd.read_csv("Y_train.csv"))
testX = pd.read_csv("X_test.csv")
testY = np.ravel(pd.read_csv("Y_test.csv"))
valX = pd.read_csv("X_val.csv")
valY = np.ravel(pd.read_csv("Y_val.csv"))

### Experimento: Regresión Lineal

En este experimento se evalúa el rendimiento del modelo **Regresión Lineal** utilizando diferentes combinaciones de hiperparámetros.  
El error que se medirá es el **Root Mean Square Error (RMSE)**, el cual permite estimar qué tan alejadas están las predicciones de los valores reales.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


X_train = pd.read_csv(r"C:\Users\Catalina\OneDrive\Escritorio\IA\Proyecto4-IA\Datos\TrainX.csv")
y_train = pd.read_csv(r"C:\Users\Catalina\OneDrive\Escritorio\IA\Proyecto4-IA\Datos\TrainY.csv")

X_test = pd.read_csv(r"C:\Users\Catalina\OneDrive\Escritorio\IA\Proyecto4-IA\Datos\TestX.csv")
y_test = pd.read_csv(r"C:\Users\Catalina\OneDrive\Escritorio\IA\Proyecto4-IA\Datos\TestY.csv")

X_val = pd.read_csv(r"C:\Users\Catalina\OneDrive\Escritorio\IA\Proyecto4-IA\Datos\ValidationX.csv")
y_val = pd.read_csv(r"C:\Users\Catalina\OneDrive\Escritorio\IA\Proyecto4-IA\Datos\ValidationY.csv")

# Si la variable objetivo tiene más de una columna
if 'Injury_Next_Season' in y_train.columns:
    y_train = y_train['Injury_Next_Season']
    y_test = y_test['Injury_Next_Season']
    y_val = y_val['Injury_Next_Season']

# ===============================
# Seleccionar las variables independientes
# ===============================
variables = ['Knee_Strength_Score', 'Reaction_Time_ms', 'Stress_Level_Score']

X_train_sel = X_train[variables]
X_test_sel = X_test[variables]
X_val_sel = X_val[variables]

# ===============================
# Entrenar el modelo de regresión lineal
# ===============================
modelo = LinearRegression()
modelo.fit(X_train_sel, y_train)

# ===============================
# Evaluar el modelo (RMSE)
# ===============================
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

train_pred = modelo.predict(X_train_sel)
test_pred = modelo.predict(X_test_sel)
val_pred = modelo.predict(X_val_sel)

print("RMSE DEL MODELO")
print(f"Entrenamiento: {rmse(y_train, train_pred):.4f}")
print(f"Prueba: {rmse(y_test, test_pred):.4f}")
print(f"Validación: {rmse(y_val, val_pred):.4f}")

# ===============================
# Tres valores por variable para crear escenarios
# ===============================
knee_values = [52.39, 74.99, 93.90]     
reaction_values = [180.0, 249.12, 306.73]
stress_values = [21.56, 54.05, 87.07]

resultados = []

for knee in knee_values:
    for reaction in reaction_values:
        for stress in stress_values:
            escenario = pd.DataFrame({
                'Knee_Strength_Score': [knee],
                'Reaction_Time_ms': [reaction],
                'Stress_Level_Score': [stress]
            })
            pred = modelo.predict(escenario)[0]
            resultados.append([knee, reaction, stress, pred])

# ===============================
# Tabla de resultados del experimento
# ===============================
tabla_resultados = pd.DataFrame(
    resultados,
    columns=['Knee_Strength_Score', 'Reaction_Time_ms', 'Stress_Level_Score', 'Predicción']
)

print("\nRESULTADOS DE LOS ESCENARIOS")
print(tabla_resultados)


RMSE DEL MODELO
Entrenamiento: 0.3615
Prueba: 0.3613
Validación: 0.3501

RESULTADOS DE LOS ESCENARIOS
    Knee_Strength_Score  Reaction_Time_ms  Stress_Level_Score  Predicción
0                 52.39            180.00               21.56   29.497892
1                 52.39            180.00               54.05   35.324118
2                 52.39            180.00               87.07   41.245385
3                 52.39            249.12               21.56   41.702172
4                 52.39            249.12               54.05   47.528398
5                 52.39            249.12               87.07   53.449665
6                 52.39            306.73               21.56   51.874171
7                 52.39            306.73               54.05   57.700397
8                 52.39            306.73               87.07   63.621664
9                 74.99            180.00               21.56   26.627015
10                74.99            180.00               54.05   32.453240
11        

### Variables independientes seleccionadas

En este experimento se usaron tres variables del conjunto de datos como entrada para el modelo de regresión lineal:

- **Knee_Strength_Score:** mide la fuerza de las rodillas. Un valor alto indica una mayor fortaleza muscular y, en general, menor probabilidad de lesión.  
- **Reaction_Time_ms:** representa el tiempo de reacción del jugador medido en milisegundos. Cuanto más alto es el valor, más lento es el tiempo de respuesta.  
- **Stress_Level_Score:** refleja el nivel de estrés del jugador. Un valor más alto puede indicar mayor tensión mental o emocional, lo que influye negativamente en el rendimiento físico.



### Resultados del modelo

El modelo de **Regresión Lineal** obtuvo los siguientes valores de error (RMSE):

- Entrenamiento: **0.3615**  
- Prueba: **0.3613**  
- Validación: **0.3501**

Estos valores son bajos y muy parecidos entre sí, lo que indica que el modelo tiene un buen desempeño y generaliza bien.  
El RMSE muestra, en promedio, cuánta diferencia hay entre las predicciones del modelo y los valores reales: cuanto más bajo sea este número, más precisas son las predicciones.
